# Explore black carbon measurements from the RDS

I made this version after comparing the CSV with the RDS. All 63 date-only records have a time of midnight in the RDS, so I’m using those timestamps here instead of treating them as missing. I kept the original notebook so we can still look back at the CSV results.

I also switched the plots to Plotly so we can hover over values, zoom into dates, and turn sensors on or off when looking at patterns.

Run the cells from top to bottom. This covers the same exploration as the CSV notebook: full-file quality checks, sensor coverage, daily means, and distributions and correlations from a reproducible sample of up to 50,000 rows. Neither source file is changed.

Requires Python packages `pandas`, `numpy`, `plotly`, and `nbformat`, plus **R with `data.table`** and `Rscript` on your PATH. R reads the full RDS into memory; Python then processes a temporary transfer file in chunks. Allow several minutes and enough memory for the RDS and disk space for a temporary file roughly the size of the CSV. The transfer file is removed after scanning.

Times are shown in UTC, as stored in the RDS. We still need to confirm how the original sensor times were assigned and what the measurement units are. Negative BC readings are kept.

This is the main analysis notebook. The CSV is an optional comparison source in `data/`; the original CSV notebook is in `archive/`. See the README for the combined findings.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import tempfile
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# If needed: %pip install pandas numpy plotly nbformat
# In R, if needed: install.packages("data.table")
REPO_DIR = next((p for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "black_carbon"]
                     if (p / "data").is_dir() and (p / "README.md").is_file()), Path.cwd())
DATA_DIR = REPO_DIR / "data"
RDS_PATH = DATA_DIR / "Black Carbon Concentrations.RDS"
CSV_PATH = DATA_DIR / "Black Carbon Concentrations.csv"  # Optional comparison only
CHUNK_SIZE = 100_000
SAMPLE_SIZE = 50_000
RANDOM_SEED = 42
NUMERIC_COLUMNS = ["Temperature", "RH", "Sense", "Reference", "Attenuation",
                   "BC", "Smoothed", "Latitude", "Longitude"]
SOURCE_COLUMNS = ["SensorID", "date", *NUMERIC_COLUMNS]
pd.set_option("display.max_columns", 30)
px.defaults.template = "plotly_white"
FIGURES = {}

def show_plot(name, fig):
    FIGURES[name] = fig
    fig.show(config={"scrollZoom": True, "displaylogo": False})

if not RDS_PATH.is_file():
    raise FileNotFoundError(f"Update DATA_DIR. Missing: {RDS_PATH}")
RSCRIPT = shutil.which("Rscript")
if not RSCRIPT:
    raise RuntimeError("Install R and put Rscript on PATH before running this notebook.")
print(f"Source: {RDS_PATH}")

## Read the original RDS timestamps
R checks the timestamp type and timezone before passing the data to Python. Dates travel as epoch seconds (seconds since 1970-01-01 UTC), which keeps midnight and avoids date-only text formatting. Source record numbers stay attached to the sampled rows and timestamp checks.

In [ ]:
R_EXPORT = r"""
args <- commandArgs(trailingOnly=TRUE)
if (!requireNamespace("data.table", quietly=TRUE)) stop("Install data.table in R first.")
x <- readRDS(args[1])
stopifnot(is.data.frame(x), inherits(x$date, "POSIXct"))
tz <- attr(x$date, "tzone")
if (!identical(tz, "UTC")) stop("Expected RDS timezone UTC; inspect metadata before continuing.")
cat("RDS rows:", nrow(x), " | timestamp class:", class(x$date), " | timezone:", tz, "\n")
x$date <- as.numeric(x$date)
data.table::fwrite(x, args[2], na="NA")
"""

transfer = tempfile.TemporaryDirectory(prefix="ucdavis_rds_")
transfer_path = Path(transfer.name) / "rds_transfer.csv"
try:
    result = subprocess.run([RSCRIPT, "-e", R_EXPORT, str(RDS_PATH), str(transfer_path)],
                            capture_output=True, text=True, check=True)
    print(result.stdout)
    preview = pd.read_csv(transfer_path, nrows=10)
    if list(preview.columns) != SOURCE_COLUMNS:
        raise ValueError("Unexpected RDS columns; inspect the source before proceeding.")
    preview["date"] = pd.to_datetime(preview["date"], unit="s", utc=True, errors="raise")
    display(preview)
    display(preview.dtypes.rename("RDS_transfer_dtype").to_frame())
except Exception:
    transfer.cleanup()
    raise

## Scan the complete file
Counts and daily means use every record. Nonfinite numeric values are counted separately and excluded from numeric summaries. Missing or invalid dates are excluded from daily summaries, but retained in total row counts.

The random-priority sampling method and seed match the original notebook. Daily means use sums and valid observation counts rather than averages of chunk means. The temporary transfer is deleted even if the scan fails; rerun the previous cell before retrying this one.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
row_count = 0
missing = pd.Series(0, index=preview.columns, dtype="int64")
invalid_numeric = pd.Series(0, index=NUMERIC_COLUMNS, dtype="int64")
nonfinite = invalid_numeric.copy()
invalid_dates = negative_bc = 0
sensor_parts, daily_parts = [], []
sample = pd.DataFrame()
midnight_parts = []

try:
    for chunk_number, chunk in enumerate(pd.read_csv(transfer_path, chunksize=CHUNK_SIZE, dtype={"SensorID": "string"}), start=1):
        chunk["source_record_number"] = np.arange(row_count + 1, row_count + len(chunk) + 1)
        row_count += len(chunk)
        missing += chunk[SOURCE_COLUMNS].isna().sum()
        raw_date = chunk["date"]
        chunk["date"] = pd.to_datetime(raw_date, unit="s", utc=True, errors="coerce")
        invalid_dates += int((raw_date.notna() & chunk["date"].isna()).sum())
        midnight_parts.append(chunk.loc[raw_date.notna() & raw_date.mod(86400).eq(0),
                                        ["source_record_number", "SensorID", "date"]].copy())
        for col in NUMERIC_COLUMNS:
            raw = chunk[col]
            values = pd.to_numeric(raw, errors="coerce").astype(float)
            invalid_numeric[col] += int((raw.notna() & values.isna()).sum())
            nonfinite[col] += int(np.isinf(values).sum())
            chunk[col] = values.replace([np.inf, -np.inf], np.nan)
        negative_bc += int(chunk["BC"].lt(0).sum())
    
        sensor_parts.append(chunk.groupby("SensorID", dropna=False).agg(
            rows=("BC", "size"), valid_BC=("BC", "count"),
            BC_sum=("BC", "sum"), first_date=("date", "min"), last_date=("date", "max")))
        daily_parts.append(chunk.assign(day=chunk["date"].dt.floor("D")).groupby(
            ["SensorID", "day"], dropna=False).agg(
            rows=("BC", "size"), valid_BC=("BC", "count"), BC_sum=("BC", "sum")))
    
        chunk["_priority"] = rng.random(len(chunk))
        sample = pd.concat([sample, chunk], ignore_index=True).nsmallest(SAMPLE_SIZE, "_priority")
        if chunk_number % 10 == 0:
            print(f"Scanned {row_count:,} rows...")
finally:
    transfer.cleanup()

if row_count == 0:
    raise ValueError("The RDS contains no data rows.")
sample = sample.drop(columns="_priority").reset_index(drop=True)
sensors = pd.concat(sensor_parts).groupby(level=0, dropna=False).agg(
    rows=("rows", "sum"), valid_BC=("valid_BC", "sum"), BC_sum=("BC_sum", "sum"),
    first_date=("first_date", "min"), last_date=("last_date", "max"))
sensors["mean_BC"] = sensors.pop("BC_sum") / sensors["valid_BC"].replace(0, np.nan)
sensors = sensors.sort_values("rows", ascending=False)
daily = pd.concat(daily_parts).groupby(level=[0, 1], dropna=False).sum().reset_index()
daily = daily.loc[daily["day"].notna()].copy()
daily["mean_BC"] = daily.pop("BC_sum") / daily["valid_BC"].replace(0, np.nan)
quality = pd.DataFrame({"missing": missing, "missing_pct": missing / row_count * 100,
                        "invalid_numeric": invalid_numeric, "nonfinite_numeric": nonfinite})
quality[["invalid_numeric", "nonfinite_numeric"]] = quality[["invalid_numeric", "nonfinite_numeric"]].fillna(0).astype(int)
print(f"Finished: {row_count:,} rows; {len(sample):,} sampled rows.")
midnight_records = pd.concat(midnight_parts, ignore_index=True)
assert int(sensors["rows"].sum()) == row_count
assert len(sample) == min(SAMPLE_SIZE, row_count)

## Check the date-only CSV records against the RDS
This optional check reads only the CSV sensor IDs and dates. It matches date-only entries to RDS midnight records by source record number, sensor ID, and calendar date. The separate full-file inspection already confirmed that the measurement values and row order match between the supplied files. If either source is replaced, repeat that full comparison before relying on row positions.

In [ ]:
timestamp_comparison = pd.DataFrame()
if CSV_PATH.is_file():
    date_only_parts = []
    offset = 0
    for raw in pd.read_csv(CSV_PATH, usecols=["SensorID", "date"], dtype="string",
                           keep_default_na=False, chunksize=CHUNK_SIZE):
        raw["source_record_number"] = np.arange(offset + 1, offset + len(raw) + 1)
        offset += len(raw)
        date_only_parts.append(raw.loc[raw["date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")].copy())
    date_only = pd.concat(date_only_parts, ignore_index=True).rename(columns={"date": "csv_date"})
    timestamp_comparison = date_only.merge(
        midnight_records.rename(columns={"date": "rds_timestamp_utc", "SensorID": "rds_sensor"}),
        on="source_record_number", how="left", validate="one_to_one")
    timestamp_comparison["confirmed_midnight"] = (
        timestamp_comparison["SensorID"].eq(timestamp_comparison["rds_sensor"])
        & timestamp_comparison["csv_date"].eq(
            timestamp_comparison["rds_timestamp_utc"].dt.strftime("%Y-%m-%d"))).fillna(False)
    assert offset == row_count, "CSV and RDS row counts differ; repeat the fidelity inspection."
    assert timestamp_comparison["confirmed_midnight"].all(), "Some records do not match; inspect before proceeding."
    print(f"{len(timestamp_comparison):,} date-only CSV records confirmed as midnight in the RDS.")
    display(timestamp_comparison)
else:
    print("CSV not found; skipped comparison. RDS exploration can still continue.")
print(f"RDS midnight records: {len(midnight_records):,}; invalid timestamps: {invalid_dates:,}")

## Full-file coverage and quality

In [ ]:
display(pd.Series({
    "total_rows": row_count,
    "identified_sensors": int(sensors.index.notna().sum()),
    "first_timestamp": sensors["first_date"].min(),
    "last_timestamp": sensors["last_date"].max(),
    "unparseable_dates": invalid_dates,
    "negative_BC_rows": negative_bc,
    "negative_BC_pct_of_valid": 100 * negative_bc / sensors["valid_BC"].sum()
        if sensors["valid_BC"].sum() else np.nan,
}).to_frame("value"))
display(quality.sort_values("missing_pct", ascending=False))
display(sensors)

fig = px.bar(quality.reset_index(names="column").sort_values("missing_pct"),
             x="missing_pct", y="column", orientation="h", hover_data=["missing"],
             title="Missing values — full RDS", labels={"missing_pct": "Percent of rows", "column": "Column"})
show_plot("missing_values", fig)

## Sample statistics and distributions
These quantiles and distributions describe the random sample. To keep negative readings and large values visible together, the x-axis uses a signed log transform: `sign(BC) × log(1 + abs(BC))`. Tick labels and hover values show the original units. The histogram bins are equally spaced in transformed coordinates; bar heights are counts, not density.

In [ ]:
display(sample[NUMERIC_COLUMNS].describe(percentiles=[.01, .05, .5, .95, .99]).T)
bc = sample["BC"].dropna().to_numpy()
if len(bc):
    transformed = np.sign(bc) * np.log1p(np.abs(bc))
    counts, edges = np.histogram(transformed, bins=80)
    original_edges = np.sign(edges) * np.expm1(np.abs(edges))
    centers = (edges[:-1] + edges[1:]) / 2
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Sample BC distribution", "Sample cumulative distribution"])
    fig.add_trace(go.Bar(x=centers, y=counts, width=np.diff(edges),
        customdata=np.column_stack([original_edges[:-1], original_edges[1:]]),
        hovertemplate="BC bin: %{customdata[0]:.3g} to %{customdata[1]:.3g}<br>Sample rows: %{y}<extra></extra>"), row=1, col=1)
    ordered = np.sort(bc)
    fig.add_trace(go.Scattergl(x=np.sign(ordered)*np.log1p(np.abs(ordered)),
        y=np.arange(1, len(ordered)+1)/len(ordered), mode="lines", customdata=ordered,
        hovertemplate="BC: %{customdata:.5g}<br>Cumulative fraction: %{y:.4f}<extra></extra>"), row=1, col=2)
    ticks = np.array([-1e6, -1e5, -1e4, -1e3, -100, -10, -1, 0, 1, 10, 100, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8])
    ticks = ticks[(ticks >= bc.min()) & (ticks <= bc.max())]
    fig.update_xaxes(tickmode="array", tickvals=np.sign(ticks)*np.log1p(np.abs(ticks)),
                     ticktext=[f"{t:g}" for t in ticks], title_text="BC (source units; signed log scale)")
    fig.update_yaxes(title_text="Sample rows", row=1, col=1)
    fig.update_yaxes(title_text="Fraction of sample", row=1, col=2)
    fig.update_layout(template="plotly_white", showlegend=False, height=450)
    show_plot("bc_distributions", fig)

## Daily patterns by sensor
Full-file daily means and counts for the five sensors with the most rows. Midnight readings are included in their UTC calendar day. Means include negative BC readings and are weighted by observations, not elapsed time. Missing days appear as gaps in the mean plot and zero counts below it. Click a legend entry to hide or show that sensor in both panels, or double-click to isolate it.

In [ ]:
top_sensors = sensors.loc[sensors.index.notna()].head(5).index
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Daily mean BC — full RDS", "Valid BC readings per day — full RDS"])
for i, sensor in enumerate(top_sensors):
    series = daily.loc[daily["SensorID"].eq(sensor)].set_index("day").sort_index()
    if series.empty:
        continue
    series = series.reindex(pd.date_range(series.index.min(), series.index.max(), freq="D"))
    color = px.colors.qualitative.Plotly[i % 10]
    for row, col in [(1, "mean_BC"), (2, "valid_BC")]:
        values = series[col] if row == 1 else series[col].fillna(0)
        fig.add_trace(go.Scatter(x=series.index, y=values, mode="lines", name=sensor,
            legendgroup=sensor, showlegend=row == 1, line={"color": color}, connectgaps=False,
            hovertemplate="%{x|%Y-%m-%d} UTC<br>%{y:.5g}<extra>%{fullData.name}</extra>"), row=row, col=1)
fig.update_yaxes(title_text="BC (source units)", row=1, col=1)
fig.update_yaxes(title_text="Valid readings", row=2, col=1)
fig.update_xaxes(title_text="Date (UTC)", rangeslider_visible=True, row=2, col=1)
fig.update_layout(template="plotly_white", height=750, hovermode="x unified", legend={"groupclick": "togglegroup"})
show_plot("daily_patterns", fig)

## Explore relationships
Sample-based Pearson correlations use pairwise complete observations. Relationships across sensors can reflect sensor differences or time trends and do not establish causation. Hover over a cell to see the coefficient.

In [ ]:
correlations = sample[NUMERIC_COLUMNS].corr()
display(correlations.round(2))
fig = go.Figure(go.Heatmap(z=correlations.to_numpy(), x=NUMERIC_COLUMNS, y=NUMERIC_COLUMNS,
    zmin=-1, zmax=1, colorscale="RdBu", reversescale=True,
    colorbar={"title": "Pearson r"},
    hovertemplate="%{y} vs %{x}<br>Pearson r: %{z:.3f}<extra></extra>"))
fig.update_layout(template="plotly_white", title="Numeric correlations — random sample", height=650,
                  xaxis={"tickangle": -45}, yaxis={"autorange": "reversed"})
show_plot("correlations", fig)

Change `SENSOR` below to another sensor ID. This plot shows sampled observations, so gaps here do not establish missing measurements in the full dataset. Use `daily` for full-file daily aggregates and `sensors` for coverage summaries. Sampled rows retain their original source record numbers.

In [ ]:
SENSOR = top_sensors[0] if len(top_sensors) else None
sensor_sample = sample.loc[sample["SensorID"].eq(SENSOR)].sort_values("date")
display(sensor_sample.head(20))
if not sensor_sample.empty:
    fig = px.scatter(sensor_sample, x="date", y="BC", hover_data=["source_record_number", "Temperature", "RH"],
        title=f"Sampled BC readings — {SENSOR}", labels={"date": "Timestamp (UTC)", "BC": "BC (source units)"})
    show_plot("sensor_sample", fig)
# display(sample.nlargest(20, "BC"))
# display(daily.loc[daily["SensorID"].eq(SENSOR)])

## What changed
The midnight readings now have valid timestamps and are included in the daily summaries. This doesn’t fill in the missing coordinates or smoothed values, since those are missing in the RDS too. It also doesn’t tell us whether Excel caused the formatting change.

The plots stay interactive when this notebook is opened in a Plotly-capable Jupyter or VS Code environment. A static preview may not show them. To share a chart separately, run `FIGURES["daily_patterns"].write_html("daily_patterns.html", include_plotlyjs=True)` for a standalone interactive HTML file.

I started with an exploratory review of the black carbon data without a specific research question, focusing on data quality, sensor coverage, and patterns in the readings.
I checked missing values, timestamp formats, negative BC readings, and daily patterns across sensors. I found 63 date-only timestamps in the CSV and compared it with the RDS. All 63 were midnight readings, and the measurement values matched between the files, so this appears to be a formatting issue.
I also created a second notebook using the RDS timestamps and added interactive Plotly graphs to make the results easier to explore. The missing coordinates and smoothed values are present in both files, so those still need investigation.
I haven’t checked for duplicate records yet. That’s my next step, along with reviewing sampling gaps and confirming how to interpret the negative BC readings.